# Getting the Data Ready for the Model

**Goal:** predict whether a person has diabetes (`Diabetes_binary`: 0 = no, 1 = yes) from health and lifestyle answers in a 2015 CDC survey.

This notebook prepares the data, trains a model, and checks how well it works:

1. **Split** the data into a training set and a test set
2. **Build** the model inputs (`X`) and the answer we want to predict (`y`)
3. **Set class weights** so the model doesn't ignore the smaller group
4. **Train** the logistic regression model
5. **Predict** on the test set
6. **Evaluate** with accuracy, precision, recall, RMSE, and R²

The plan for the model itself: use **logistic regression**, a classic model for yes/no questions.

## Load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("diabetes_binary_health_indicators_BRFSS2015.csv")

# List of variables that are logically binary/categorical (factor in R)
binary_vars = [
    "Diabetes_binary", "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex",
]

# Convert those columns to pandas 'category' dtype (equivalent of as.factor)
for col in binary_vars:
    df[col] = df[col].astype("category")

print(df.head())
print(df.dtypes)

## Step 1: Split into training and test sets

A model should be judged on data it has **never seen**. 

So we split the data:
- **Training set (80%)**: the model learns from this.
- **Test set (20%)**: we hide it and use it at the end to check the model.

In [ ]:
diabetic_train, diabetic_test = train_test_split(
    df, test_size=0.2, stratify=df["Diabetes_binary"], random_state=1
)

What the settings mean:
- `test_size=0.2`: 20% of rows go to the test set
- `stratify=df["Diabetes_binary"]`: both sets get the same share of people with diabetes
- `random_state=1`: the "random" shuffle is the same every time, so everyone gets the same split

Let's check it worked:

In [ ]:
print("Training rows:", len(diabetic_train))
print("Test rows:    ", len(diabetic_test))
print()
print("Share with diabetes in each set:")
print("Training:", diabetic_train["Diabetes_binary"].value_counts(normalize=True).round(3).to_dict())
print("Test:    ", diabetic_test["Diabetes_binary"].value_counts(normalize=True).round(3).to_dict())

The shares are nearly identical in both sets. That is what `stratify` does.

## Correlation Heatmap
Shows the strength and direction of relationships between the variables below. REMEMBER:
- Values closer to 1 indicate strong relationship
- Values closer to -1 indicate strong inverse relationship
- Values closer to 0 indicate weak relationship

In [ ]:
correlation_columns = [
    "Diabetes_binary", "HighBP", "HighChol", "BMI", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Age", "HvyAlcoholConsump", "Education"
]

corr_data = df[correlation_columns].astype(float)
corr_matrix = corr_data.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f"
)
plt.title("Correlation Between Health Indicators")
plt.tight_layout()
plt.show()

## Step 2: Build the model inputs (`X`) and the answer (`y`)

- **`X`**: the columns the model uses to make a prediction (the *predictors*)
- **`y`**: the answer we want it to predict

Models only understand numbers, so yes/no category columns get converted with `pd.get_dummies`. For example, `HighBP` (0 or 1) becomes a single yes/no column called `HighBP_1.0` (shown as True/False, which the model treats as 1/0). The `drop_first=True` setting removes the extra, redundant column (if `HighBP_1.0` is 0, we already know the person is in the "no" group).

In [ ]:
# Build the design (model) matrix for the chosen predictors
predictors = [
    "HighBP", "HighChol", "HvyAlcoholConsump", "Stroke", "DiffWalk",
    "Age", "Income", "CholCheck", "GenHlth", "PhysActivity",
]

X_train = pd.get_dummies(diabetic_train[predictors], drop_first=True)
y_train = diabetic_train["Diabetes_binary"].astype(int)

X_test = pd.get_dummies(diabetic_test[predictors], drop_first=True)
# Make sure the test columns match the training columns exactly
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
y_test = diabetic_test["Diabetes_binary"].astype(int)

What the model will actually see:

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
print()
X_train.head()

Each row is one person, and each column is one piece of information about them. `y_train` holds the matching answers (0 or 1).

We also make sure `X_test` has the **same columns in the same order** as `X_train`, because the model expects its inputs to look the same every time.

## Step 3: Class weights

Most people in the survey do **not** have diabetes. If the model sees mostly "no" answers, it can score well by almost always guessing "no" while missing the people we care about most.

**Class weights** fix this: each person in the smaller group counts for more during training, so both groups matter equally.

In [ ]:
class_counts = y_train.value_counts()
inv_weights = 1 / class_counts
sample_weights = y_train.map(inv_weights)

How it works: the weight for each class is `1 / (number of people in that class)`. The smaller group gets the bigger weight. Let's see:

In [ ]:
summary = pd.DataFrame({
    "people": class_counts,
    "weight_per_person": inv_weights,
})
summary["total_weight"] = summary["people"] * summary["weight_per_person"]
summary

Look at the `total_weight` column: each group adds up to **1**. Even though one group is much bigger, both now have equal influence on the model.

## Step 4: Train the logistic regression model

Now we use `X_train`, `y_train`, and `sample_weights` to train the model.

**Logistic regression** gives each person a probability (0 to 1) of having diabetes. It looks at all the inputs and learns how much each one pushes the probability up or down.

One small adjustment first: our weights from Step 3 are very tiny numbers (each group adds up to 1). Logistic regression works best when the average weight is about 1, so we scale them up. The *relative* sizes don't change: the smaller group still counts more.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Scale the weights so the average weight is 1 (the smaller group still counts more)
scaled_weights = sample_weights * len(y_train) / 2

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train, sample_weight=scaled_weights)

print("Model trained!")

## Step 5: Make predictions

The model gives a **probability** for each person in the test set. We turn that into a yes/no prediction using a **cutoff**: 0.45 or higher means we predict "diabetes".

In [ ]:
pred_probs = model.predict_proba(X_test)[:, 1]   # probability of diabetes
pred_class = (pred_probs >= 0.45).astype(int)    # 1 = diabetes, 0 = no diabetes

# Look at the first 5 predictions next to the real answers
pd.DataFrame({
    "probability": pred_probs[:5].round(3),
    "predicted": pred_class[:5],
    "actual": y_test.values[:5],
})

## Step 6: Accuracy vs. Precision vs. Recall

**Classification scores** (these use the yes/no predictions):
- **Accuracy**: out of everyone, how many did the model get right?
- **Precision**: of the people the model flagged as having diabetes, how many really do?
- **Recall**: of the people who really have diabetes, how many did the model catch?

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
)

# Classification scores (from the yes/no predictions)
accuracy = accuracy_score(y_test, pred_class)
precision = precision_score(y_test, pred_class)
recall = recall_score(y_test, pred_class)

print("Accuracy vs. Precision vs. Recall")
print(f"  Accuracy : {accuracy:.3f}")
print(f"  Precision: {precision:.3f}")
print(f"  Recall   : {recall:.3f}")

**How to read these:** accuracy alone can look fine even if the model misses many people with diabetes, so check **recall** too. Precision tells you how many of the model's "diabetes" flags are correct. A low or negative R² here doesn't mean the model is bad: the class weights push the probabilities upward, so they don't line up well with the true 0/1 answers.

## Recap

| Step | What we did | Why |
|------|-------------|-----|
| 1 | Split into training (80%) and test (20%) | Judge the model on data it hasn't seen |
| 2 | Built `X` (inputs) and `y` (answers) | Models need numbers in a consistent layout |
| 3 | Made class weights | Keep the model from ignoring the smaller group |
| 4 | Trained a logistic regression | Learn the link between the inputs and diabetes |
| 5 | Predicted probabilities, then yes/no at a 0.45 cutoff | Turn the model's output into answers we can check |
| 6 | Printed accuracy, precision, recall, RMSE, and R² | See how well the model does on the test set |